# Segment-wise (partial) deepfake **audio** fine-tuning — Kaggle

The stock audio model is a LoRA adaptation over ASVspoof / WaveFake, where **every**
**file is entirely real or entirely fake**. So it learns an *utterance-level* decision
and tends to flag a tampered clip end-to-end — "it falsifies the whole thing".

To make detection **segment-wise** the model has to see clips where real and fake audio
coexist, with per-segment ground truth. This notebook:

1. builds whole-file manifests from a spoof dataset already on Kaggle (ASVspoof / WaveFake),
2. **synthesizes partial-spoof clips** (`scripts/make_partialspoof.py`) with exact fake
   intervals,
3. **refines your existing model** on those segment labels
   (`scripts/train_audio_segments.py`), and
4. exports a `merged/` dir you drop straight into `configs/default.yaml`.

> Enable **GPU** (Settings ▸ Accelerator ▸ GPU T4/P100) and **Internet** (needed to pull
> the base model weights).

## 1. Get the code + install deps

In [ ]:
# If you added this repo as a Kaggle 'Utility Script' / dataset, skip the clone.
import os, sys, subprocess
REPO = '/kaggle/working/deepfake-detection'
if not os.path.isdir(REPO):
    subprocess.run(['git','clone','--depth','1',
                    'https://github.com/atharv-0509/deepfake-detection.git', REPO], check=True)
os.chdir(REPO)
sys.path.insert(0, REPO)
print('cwd:', os.getcwd())

In [ ]:
# Kaggle images already ship torch/transformers. Add the training extras.
!pip -q install 'peft>=0.10' 'accelerate>=0.30' soundfile librosa scikit-learn 2>/dev/null
import torch; print('cuda:', torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else '')

## 2. Build whole-file manifests

Point these at whatever spoof dataset you've attached to the notebook. Two common
Kaggle sources are shown — **run the one you have** and skip the other.

In [ ]:
# --- Option A: ASVspoof 2019 LA -------------------------------------------
# Attach an ASVspoof 2019 LA dataset, then set ASV_ROOT to the dir that contains
# ASVspoof2019_LA_train/, ..._dev/, ..._cm_protocols/.
ASV_ROOT = '/kaggle/input/asvspoof-2019/LA'   # <-- edit to your dataset path
import os
if os.path.isdir(ASV_ROOT):
    !python scripts/prepare_asvspoof.py --root "$ASV_ROOT" --out-dir data/asvspoof --splits train dev
    SRC_TRAIN, SRC_DEV = 'data/asvspoof/asvspoof_train.csv', 'data/asvspoof/asvspoof_dev.csv'
    print('ASVspoof manifests ready')
else:
    print('ASV_ROOT not found — edit the path or use Option B below')

In [ ]:
# --- Option B: WaveFake + LJSpeech ----------------------------------------
WF_ROOT = '/kaggle/input/wavefake'                        # <-- edit
LJ_ROOT = '/kaggle/input/ljspeech-11/LJSpeech-1.1'        # <-- edit
import os
if os.path.isdir(WF_ROOT) and os.path.isdir(LJ_ROOT):
    !python scripts/prepare_wavefake.py --wavefake-root "$WF_ROOT" --ljspeech-root "$LJ_ROOT" \
        --out-dir data/wavefake --per-vocoder-cap 1500 --balance
    SRC_TRAIN, SRC_DEV = 'data/wavefake/wavefake_train.csv', 'data/wavefake/wavefake_dev.csv'
    print('WaveFake manifests ready')
else:
    print('WaveFake/LJSpeech not found — use Option A above')

## 3. Synthesize partial-spoof clips (segment-level labels)

Each synthesized clip splices bonafide + spoof chunks and records exactly which
time intervals are fake. Pass-through originals keep some fully-real / fully-fake
examples so the model doesn't forget utterance-level cues. Tune `--n-partial` to
your time budget.

In [ ]:
!python scripts/make_partialspoof.py \
    --manifest $SRC_TRAIN \
    --out-dir data/partialspoof/train --name partial_train \
    --n-partial 6000 --passthrough-real 2000 --passthrough-fake 2000 --seed 42

!python scripts/make_partialspoof.py \
    --manifest $SRC_DEV \
    --out-dir data/partialspoof/dev --name partial_dev \
    --n-partial 1000 --passthrough-real 400 --passthrough-fake 400 --seed 7

In [ ]:
# Peek at the segment manifest — note the per-clip fake_segments column.
import pandas as pd
df = pd.read_csv('data/partialspoof/train/partial_train.csv')
print(df['source'].value_counts())
df[df.source=='partial'].head(5)

## 4. Train segment-wise (LoRA, refining your existing model)

`--model-id` can be the stock base **or** a `merged/` dir from a previous
`train_audio.py` run — the segment adapter stacks on top. Pass several
`--train-csv` to combine partial data from multiple datasets.

In [ ]:
BASE = 'MelodyMachine/Deepfake-audio-detection'   # or a prior merged/ dir to refine
!python scripts/train_audio_segments.py \
    --model-id "$BASE" \
    --train-csv data/partialspoof/train/partial_train.csv \
    --dev-csv   data/partialspoof/dev/partial_dev.csv \
    --output-dir /kaggle/working/audio-partial-lora \
    --epochs 5 --batch-size 32 --lr 3e-4 --lora-r 16 --fp16 \
    --segment-seconds 1.0 --positive-overlap 0.5

## 5. Sanity check — score a synthesized clip window-by-window

The fine-tuned model should now give a **high** fake probability on the spliced
windows and a **low** one on the real windows of the same file, instead of one
verdict for the whole clip.

In [ ]:
import numpy as np, soundfile as sf, torch, pandas as pd
from transformers import AutoFeatureExtractor, AutoModelForAudioClassification
from src.training.segment_labeling import parse_fake_segments, window_label

MODEL = '/kaggle/working/audio-partial-lora/merged'
ex = AutoFeatureExtractor.from_pretrained(MODEL)
m = AutoModelForAudioClassification.from_pretrained(MODEL).eval()
dev = 'cuda' if torch.cuda.is_available() else 'cpu'; m.to(dev)

row = pd.read_csv('data/partialspoof/dev/partial_dev.csv')
row = row[row.source=='partial'].iloc[0]
wav, sr = sf.read(row.file_path, dtype='float32', always_2d=False)
fakes = parse_fake_segments(row.fake_segments)
print('file:', row.file_path, '| fake intervals:', fakes)

win = int(1.0*sr)
for i in range(0, len(wav)-int(0.1*sr), win):
    seg = wav[i:i+win]
    if len(seg) < win: seg = np.pad(seg, (0, win-len(seg)))
    inp = ex(seg, sampling_rate=sr, return_tensors='pt').to(dev)
    with torch.no_grad():
        p = torch.softmax(m(**inp).logits, -1)[0].cpu().numpy()
    t0, t1 = i/sr, (i+win)/sr
    gt = window_label(t0, t1, fakes)
    print(f'  {t0:4.1f}-{t1:4.1f}s  P(spoof)={p[1]:.3f}  ground-truth={"FAKE" if gt else "real"}')

## 6. Use it in the pipeline

Copy `/kaggle/working/audio-partial-lora/merged` out via the notebook **Output**,
then point inference at it:

```yaml
# configs/default.yaml
audio:
  model_id: /path/to/audio-partial-lora/merged
```

`scripts/run_inference.py` and `app.py` pick it up with no code changes — every 1s
window is now scored on its own merits, so you get a real tamper timeline instead of
a single whole-clip verdict.